In [1]:
!pip install -q transformers sentencepiece accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 11.9 MB/s eta 0:00:00


In [2]:
import json
import os
import torch
from tqdm import tqdm
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
PROJECT_ROOT = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "ProfessionAI_AIengineering/9. Generative AI/"
    "Project_Generative_AI"
)

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
CAPTIONS_DIR = os.path.join(DATA_DIR, "captions")
TEXT_VARIATIONS_DIR = os.path.join(DATA_DIR, "text_variations")

os.makedirs(TEXT_VARIATIONS_DIR, exist_ok=True)

CAPTION_FILE = os.path.join(CAPTIONS_DIR, "captions_train_small.json")
TEXT_VARIATION_FILE = os.path.join(TEXT_VARIATIONS_DIR, "text_variations_train_small.json")

In [6]:
INSTALL_DEPS = True   # set to False after first successful run

if INSTALL_DEPS:
    !pip install -r "$PROJECT_ROOT/requirements.txt"

In [7]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


# Load Captions

In [8]:
with open(CAPTION_FILE, "r") as f:
  captions_dict = json.load(f)

In [9]:
captions_dict['981']['captions']

['a birman cat is sitting on the railing of a deck',
 'a birman cat is sitting on a fence']

In [10]:
captions_dict

{'0': {'class_name': 'Pomeranian',
  'captions': ['a pomeranian dog this dog is sitting on the bed',
   'a pomeranian dog my dog is sitting on the bed']},
 '1': {'class_name': 'Havanese',
  'captions': ['a havanese dog is sitting on a tennis court',
   'a havanese dog is sitting on the tennis court']},
 '2': {'class_name': 'British Shorthair',
  'captions': ['a british shorthair cat is sitting in a cardboard box',
   'a british shorthair cat is sitting in a box']},
 '3': {'class_name': 'Samoyed',
  'captions': ['a samoyed dog is sitting on the ground with his tongue out',
   'a samoyed dog is looking at the camera']},
 '4': {'class_name': 'Siamese',
  'captions': ['a siamese cat sitting on a bed',
   'a siamese cat sitting on a bed']},
 '5': {'class_name': 'Keeshond',
  'captions': ['a keeshond dog is standing on the grass',
   'a keeshond dog is standing on the grass']},
 '6': {'class_name': 'Chihuahua',
  'captions': ['a chihuahua dog is a small dog with a long body and short legs',


In [12]:
captions_dict['1022']

{'class_name': 'Pomeranian',
 'captions': ['a pomeranian dog is sitting on the edge of the dock with its back to the water and its front facing the water',
  'a pomeranian dog person is sitting on the edge of the water']}

# Load FLAN-T5-Large Model

In [ ]:
model_name = "google/flan-t5-large"

tokenizer = T5Tokenizer.from_pretrained(model_name)

model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1024, out_features=2816, bias=False)
              (wi_1): Linear(in_features=1024, out_features=2816, bias=False)
       

# Define Prompt

In [ ]:
def build_prompt(caption):
  return f"Rewrite this caption in three different ways: {caption}"

In [ ]:
def generate_variations(prompt, num_variations=3):
  inputs = tokenizer(prompt, return_tensors = "pt", truncation=True).to(device)

  outputs = model.generate(**inputs,
                          max_new_tokens=40,
                          # do_sample=True,
                          # temperature=0.7,
                          # top_k=50,
                          # top_p=0.9,
                          # # num_beams=5,
                          # num_return_sequences=num_variations

                          # num_beams=6,
                          # num_beam_groups=3,
                          # diversity_penalty=0.5,
                          # num_return_sequences=num_variations,
                          # early_stopping=True,
                          # trust_remote_code=True
                          do_sample=True,
                          temperature=0.6,
                          top_p=0.85,
                          num_return_sequences=num_variations
                          )

  decoded = [
      tokenizer.decode(output, skip_special_tokens=True)
      for output in outputs
  ]

  return decoded

In [ ]:
for img, data in list (captions_dict.items())[:10]:
  captions = list(set(data["captions"])) # remove duplicates

  for caption in captions:
    prompt = build_prompt(caption)
    # print(prompt)
    raw = generate_variations(prompt)

    print("Original:", caption)
    print("Raw generated:", raw)

Original: a pomeranian dog my dog is sitting on the bed
Raw generated: ['someone wussiest getting to sit. my white squonzy girl poke an adorable little eye. the dogs', 'man feeding and cleaning baby pups standing tall', 'three female polar ters cats sleep, sleeping against me against a wall, and watching shows ewww cute animals living side together through dog videos!my name should read sweet smile and you']


KeyboardInterrupt: 

In [ ]:
test_prompt = "Rewrite: A dog is sitting on a bed."

inputs = tokenizer(test_prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=20)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

A dog is sitting on a bed.


In [ ]:
!nvidia-smi

Thu Feb 12 18:19:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:

model_name = "google/flan-t5-xl"
tokenizer = T5Tokenizer.from_pretrained(model_name)

model = T5ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,     # REQUIRED for T4
    device_map="auto",             # automatic GPU placement
    low_cpu_mem_usage=True
)

model.eval()

print("Model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully.


In [ ]:
def build_prompt(caption):
    return f"Rewrite this caption in three different ways: {caption}"

In [ ]:
def generate_variations(prompt, num_variations=3):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        num_beams=5,
        num_return_sequences=num_variations,
        early_stopping=True
        # do_sample=True,
        # temperature=0.7,
        # top_k=50,
        # top_p=0.9,
        # # num_beams=5,
        # num_return_sequences=num_variations
    )

    return [
        tokenizer.decode(o, skip_special_tokens=True)
        for o in outputs
    ]

In [ ]:
test_caption = "a havanese dog is sitting on a tennis court"

prompt = build_prompt(test_caption)
variations = generate_variations(prompt)

print("Original:", test_caption)
print("Generated:", variations)

Original: a havanese dog is sitting on a tennis court
Generated: ['it comes down down the field one is tennis one downs the country on court', "torches glisten in warm shades across one person' womb who stands amid lighted rocks among hundreds which lie unlovened at desert gardens above ocean at low sun of this early", 'In another country two cats drib-digge at night, on what at to date one the few beaches without water tanks they choose! the little dogs and horses on which most play sport sit']


In [ ]:
for img, data in list (captions_dict.items())[:10]:
  captions = list(set(data["captions"])) # remove duplicates

  for caption in captions:
    prompt = build_prompt(caption)
    # print(prompt)
    raw = generate_variations(prompt)

    print("Original:", caption)
    print("Raw generated:", raw)

Original: a pomeranian dog this dog is sitting on the bed
Raw generated: ['He’l play basketball during workout ses', '', 'An alert male German cobre horse is pointing']
Original: a pomeranian dog my dog is sitting on the bed
Raw generated: ['meering is in sleep like always before us', 'that red one my do', 'of dogs']
Original: a havanese dog is sitting on the tennis court
Raw generated: ['Tennis Court where it feels right by someone having pizza near of someone by on its seat by some other dog. the woman doesnld like him even she like dogs to people walking near where man walking', 'as they serve off white- ball against string one another until all scoreline tied as scorecard turns as court hasnar blue wind. The players try on blue court has in fact score board and', 'He gets in two legs race race to start for an afternoon with two goals scored on different grounds of each round match at different levels to raise confidence while on top during summertime game on open area or']
Original

KeyboardInterrupt: 

# Load Mistral 7B Instruct

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# Define 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

model.eval()

print("Mistral loaded successfully.")


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Mistral loaded successfully.


In [ ]:
!nvidia-smi

Thu Feb 12 20:20:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P0             28W /   70W |   12775MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
def build_prompt(caption):
    return f"""<s>[INST]
Rewrite the caption in two different ways.
Keep the meaning the same.

Caption: {caption}
[/INST]"""

In [ ]:
def generate_variations(prompt):

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
      outputs = model.generate(
          **inputs,
          max_new_tokens=40,
          do_sample=True,
          temperature=0.7,
          top_p=0.9,
          num_return_sequences=1,
          pad_token_id=tokenizer.eos_token_id
      )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = text.split("[/INST]")[-1].strip()

    # Split numbered lines into clean list
    variations = []
    for line in response.split("\n"):
        line = line.strip()
        if len(line) > 5:
            if line[0].isdigit():
                line = line.split(".", 1)[-1].strip()
            variations.append(line)

    return variations[:2]  # ensure exactly 2


In [ ]:
# test_caption = "a havanese dog is sitting on a tennis court"

# prompt = build_prompt(test_caption)
# variations = generate_variations(prompt)

# print("Original:", test_caption)
# print("Generated:")
# for v in variations:
#     print("-", v)

In [ ]:
# captions_dict['975']

In [ ]:
text_variations = {}

for img, data in tqdm(captions_dict.items()):

    class_name = data["class_name"]
    captions = list(set(data["captions"]))  # remove duplicates

    all_generated = []

    for caption in captions:
        prompt = build_prompt(caption)
        variations = generate_variations(prompt)
        all_generated.extend(variations)

    # Remove duplicates across captions
    all_generated = list(set(all_generated))

    text_variations[img] = {
        "class_name": class_name,
        "original_captions": captions,
        "generated_captions": all_generated
    }


100%|██████████| 1104/1104 [1:42:07<00:00,  5.55s/it]


In [ ]:
import json

with open(TEXT_VARIATION_FILE, "w") as f:
    json.dump(text_variations, f, indent=4)

print("Saved successfully:", len(text_variations))

Saved successfully: 1104


Consider
- executing in batch
- add save checkpoints